# Compare neuron-group PCA scores
Compare Motor, Sensory, Sensorimotor and None with each other and with the
**full original neuron set**, using matched sessions and conditions. This notebook
runs independently of `neuron_type_pca.ipynb`; it loads the saved extraction and
fits all populations using the same preprocessing, observations and PC predictors.

Color identifies neuron type in the common plots: **Sensorimotor light green,
Motor blue, Sensory yellow, None gray**. Tracking and playback have separate
panels. In difference matrices, group labels carry these colors; the cell color
still represents the signed numerical difference.

Run top to bottom with the project's `.venv` kernel. Data paths work from the
workspace, repository or notebook directory. Results are saved under a separate
`neuron_group_comparison_...` run directory. Source data and single-type PCA runs
are not changed.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from importlib import reload
import sklearn

cwd = Path.cwd().resolve()
repo_dir = next(
    (candidate for parent in (cwd, *cwd.parents)
     for candidate in (parent, parent / "etheremin_project")
     if (candidate / "movement_bouts" / "utils.py").is_file()), None,
)
if repo_dir is None:
    raise FileNotFoundError("Run from the eTheremin workspace or inside the repository.")
sys.path.insert(0, str(repo_dir / "movement_bouts"))
import utils
import evaluation
reload(utils)
reload(evaluation)
from utils import prepare_pca_input, fit_pca_by_condition

data_dir = repo_dir.parent / "data"
bout_dir = data_dir / "derived" / "movement_bouts"
premotor_dir = data_dir / "derived" / "premotor_activity"

## 1. Settings and input tables
Choose the groups, minimum selected-type count, and `raw` versus
`baseline_subtracted` firing rates. Baseline correction subtracts each bout/unit's
mean rate over [-1, -0.7) s. Five analysis windows over [-0.5, 0) s give 100 ms
rates. No Gaussian smoothing, z-scoring or whitening is applied to PCA inputs.
Use the same PC predictors for every group; the default is PC1-PC3.

In [ ]:
COMPARISON_GROUPS = ["Motor", "Sensory", "Sensorimotor", "None"]
MIN_NEURONS = 5             # Minimum neurons of each type per session.
ACTIVITY_MODE = "raw"  # "raw" or "baseline_subtracted"
ANALYSIS_WINDOW_S = (-0.5, 0.0)
BASELINE_WINDOW_S = (-1.0, -0.7)
N_BINS = 5                # 5 = 100 ms, 2 = 250 ms, 1 = 500 ms over [-0.5, 0).
EXTRACTION_SUBDIR = "extracted_-1.2_0"
SAVE_RESULTS = True

# Pooled evaluations use PC1-PC3 by default, independently per session/condition.
EVALUATION_COLUMNS = ["PC1", "PC2", "PC3"]
DESTINATION_MIN_DISPLACEMENT = None  # 5.0 reproduces the earlier abs(end-start)>5 analysis.
DIRECTION_C = 1.0                   # Inverse L2 penalty strength.
LINEAR_SCORE = "r2_adjusted"        # Or "r2".

# These colors identify neuron types in both tracking and playback panels.
GROUP_COLORS = dict(evaluation.NEURON_TYPE_COLORS)

In [ ]:
extraction_dir = premotor_dir / EXTRACTION_SUBDIR
extraction_parameters = json.loads((extraction_dir / "parameters.json").read_text(encoding="utf-8"))
# Bout IDs are meaningful only for the exact source tables used for extraction.
for filename, expected_hash in extraction_parameters["source_sha256"].items():
    actual_hash = hashlib.sha256((bout_dir / filename).read_bytes()).hexdigest()
    if actual_hash != expected_hash:
        raise ValueError(f"{filename} has changed since extraction; regenerate aligned counts first.")
movement_bouts = pd.read_pickle(bout_dir / "movement_bouts.pkl")
session_info = pd.read_pickle(bout_dir / "session_info.pkl")
neuron_ids_by_session = pd.read_pickle(extraction_dir / "neuron_ids_by_session.pkl")
classification_path = data_dir / "df_clus_encoding_phd.pkl"
classification_table = pd.read_pickle(classification_path)
DT = extraction_parameters["DT"]
print(f"{len(movement_bouts):,} reference bouts; {len(session_info)} sessions; DT={DT:g} s.")

In [ ]:
category_to_cluster = {"Motor": 0, "Sensory": 1, "Sensorimotor": 2, "None": 3}
reference_group = "All neurons"
if len(COMPARISON_GROUPS) != len(set(COMPARISON_GROUPS)) or not COMPARISON_GROUPS:
    raise ValueError("Choose a nonempty list of distinct neuron types.")
if not set(COMPARISON_GROUPS).issubset(category_to_cluster):
    raise ValueError("Unknown comparison neuron type.")
if not isinstance(MIN_NEURONS, (int, np.integer)) or isinstance(MIN_NEURONS, bool) or MIN_NEURONS < 1:
    raise ValueError("MIN_NEURONS must be a positive integer.")
if ACTIVITY_MODE not in ("raw", "baseline_subtracted"):
    raise ValueError("ACTIVITY_MODE must be raw or baseline_subtracted.")
if (not EVALUATION_COLUMNS or len(set(EVALUATION_COLUMNS)) != len(EVALUATION_COLUMNS)
        or any(not name.startswith("PC") or not name[2:].isdigit() or int(name[2:]) < 1
               for name in EVALUATION_COLUMNS)):
    raise ValueError("Select distinct PCA columns such as PC1, PC2, PC3.")
if classification_table["neuron_idx"].isna().any() or classification_table["neuron_idx"].duplicated().any():
    raise ValueError("Classification neuron_idx must be nonmissing and unique.")
if not classification_table["cluster_encoding"].isin(category_to_cluster.values()).all():
    raise ValueError("Unknown or missing cluster_encoding.")
cluster_by_neuron = classification_table.set_index("neuron_idx")["cluster_encoding"]
comparison_settings = {
    "neuron_type": "comparison", "cluster_encoding": None,
    "min_neurons": MIN_NEURONS, "DT": DT, "N_BINS": N_BINS,
    "activity_mode": ACTIVITY_MODE, "analysis_window_s": list(ANALYSIS_WINDOW_S),
    "baseline_window_s": list(BASELINE_WINDOW_S) if ACTIVITY_MODE == "baseline_subtracted" else None,
    "extraction_directory": str(extraction_dir), "extraction_parameters": extraction_parameters,
    "classification_file": str(classification_path),
    "classification_sha256": hashlib.sha256(classification_path.read_bytes()).hexdigest(),
    "category_to_cluster": category_to_cluster.copy(),
    "standardized": False, "whiten": False, "sklearn_version": sklearn.__version__,
    "groups": [reference_group, *COMPARISON_GROUPS], "columns": list(EVALUATION_COLUMNS),
    "destination_min_displacement": DESTINATION_MIN_DISPLACEMENT, "direction_C": DIRECTION_C,
    "session_matching": "pairwise_same_idx_and_condition", "scoring": "descriptive_in_sample",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}
position_limits = (float(movement_bouts["initial_position"].min()),
                   float(movement_bouts["initial_position"].max()))

## 2. Fit each population on the same observations
Fit **all four neuron types plus All neurons** using the settings above. It recomputes the full-session
PCA with **exactly the same** time windows, normalization, bouts and evaluation
settings. Thus an older all-neuron result with a different baseline or destination
filter cannot enter the comparison accidentally. All neurons includes units with
missing classifications as well as the four assigned categories.

Each type must meet the same minimum neuron cutoff. Full-session references need
only enough neurons for the selected PCs. Every comparison matches **session idx
and condition**, including only successful finite scores. Types are compared on
their **pairwise shared sessions**; a session need not qualify for all types.
At cutoff 5, no session qualifies for all four types; Sensory has only 4 eligible
sessions. Pair counts are therefore displayed explicitly rather than comparing
means from different sets of sessions.

Top plots compare a type's score with its own full-session score; bottom plots
show type minus full-session differences (bars = paired mean). The matrices
compare every pair of types, including the full-session reference. Positive cells
favor the row group. Scores remain **descriptive and in-sample**, using the same
number of PC predictors for every group. Groups contain different numbers of
neurons: this comparison does not control population size. Neuron counts, matched
session IDs, original observation IDs and fitted models remain in the saved results.

Run the following cells after choosing preprocessing above. The extraction is loaded once, prepared once per session,
and then the same rows are used for every eligible population. Single-type PCA runs and results remain separate.

In [ ]:
comparison_groups = comparison_settings["groups"]
comparison_min_neurons = comparison_settings["min_neurons"]
comparison_group_ids = {group: {} for group in comparison_groups}
comparison_eligibility_rows = []
required_pcs = max(int(column.removeprefix("PC")) for column in comparison_settings["columns"])
for idx in session_info.index:
    all_ids = list(neuron_ids_by_session.loc[idx, "neuron_ids"])
    clusters = pd.Series(all_ids).map(cluster_by_neuron)
    for group in comparison_groups:
        ids = all_ids if group == reference_group else [
            neuron_id for neuron_id, cluster in zip(all_ids, clusters)
            if cluster == category_to_cluster[group]
        ]
        comparison_group_ids[group][idx] = ids
        minimum = required_pcs if group == reference_group else max(comparison_min_neurons, required_pcs)
        comparison_eligibility_rows.append(dict(
            group=group, idx=idx, n_neurons=len(ids), eligible=len(ids) >= minimum,
        ))
comparison_eligibility = pd.DataFrame(comparison_eligibility_rows).set_index(["group", "idx"])
display(comparison_eligibility.groupby("group").agg(
    eligible_sessions=("eligible", "sum"), median_neurons=("n_neurons", "median"),
).reindex(comparison_groups))
comparison_pca_groups = {
    group: {"settings": {**comparison_settings, "neuron_type": group}, "sessions": {},
            "session_info": session_info.copy(), "selected_neuron_ids": {},
            "position_limits": position_limits}
    for group in comparison_groups
}
neuronal_bouts = pd.read_pickle(Path(comparison_settings["extraction_directory"]) / "neuronal_bouts.pkl")
for idx in session_info.index:
    eligible_groups = [group for group in comparison_groups if comparison_eligibility.loc[(group, idx), "eligible"]]
    bout_ids = movement_bouts.index[movement_bouts["idx"].eq(idx)]
    if not eligible_groups or not bout_ids.isin(neuronal_bouts.index).any():
        continue
    session_input, session_labels = prepare_pca_input(
        neuronal_bouts, movement_bouts, neuron_ids_by_session,
        idx=idx, dt=comparison_settings["DT"], n_bins=comparison_settings["N_BINS"],
        extraction_window_s=comparison_settings["extraction_parameters"]["window_s"],
        analysis_window_s=comparison_settings["analysis_window_s"],
        baseline_window_s=comparison_settings["baseline_window_s"], activity_mode=comparison_settings["activity_mode"],
    )
    for group in eligible_groups:
        ids = comparison_group_ids[group][idx]
        fits = fit_pca_by_condition(session_input.loc[:, ids], session_labels)
        if fits:
            comparison_pca_groups[group]["sessions"][idx] = fits
            comparison_pca_groups[group]["selected_neuron_ids"][idx] = ids
    print(f"idx={idx}: fitted {', '.join(eligible_groups)}")
del neuronal_bouts
for group, cache in comparison_pca_groups.items():
    cache["session_info"] = session_info.loc[list(cache["sessions"])].copy()
print({group: len(cache["sessions"]) for group, cache in comparison_pca_groups.items()}, "fitted sessions")

## 3. Score starting position, final position and direction
Fit pooled linear regressions for initial/final position and balanced L2 logistic
regression for movement direction. Full-session/type pairs must use identical
scored observation IDs. Failed and ineligible subsets remain explicit.
These are the same descriptive, in-sample methods as the other PCA notebook;
no significance test or held-out performance estimate is introduced.

In [ ]:
comparison_evaluations = {target: {} for target in ("initial_position", "final_position", "initial_direction")}
full_index = pd.MultiIndex.from_product([session_info.index, [0, 1]], names=["idx", "condition"])
for group, cache in comparison_pca_groups.items():
    for target in comparison_evaluations:
        if cache["sessions"]:
            if target == "initial_direction":
                bundle = evaluation.run_direction_logistics(
                    cache, columns=comparison_settings["columns"], C=comparison_settings["direction_C"],
                )
            else:
                bundle = evaluation.run_label_regressions(
                    cache, target=target, columns=comparison_settings["columns"],
                    min_displacement=comparison_settings["destination_min_displacement"] if target == "final_position" else None,
                )
        else:
            # A missing population is shown explicitly rather than disappearing.
            template = comparison_evaluations[target][reference_group]
            bundle = {"summary": template["summary"].iloc[:0].copy(), "fits": {},
                      "settings": {**template["settings"], "pca_settings": cache["settings"]}}
        summary = bundle["summary"].reindex(full_index)
        absent = summary["status"].isna()
        summary.loc[absent, "status"] = "skipped"
        summary.loc[absent, "reason"] = "Neuron cutoff not met, or no usable extracted observations/PCA"
        summary["n_neurons"] = [len(comparison_group_ids[group][idx]) for idx in summary.index.get_level_values("idx")]
        bundle["summary"] = summary
        comparison_evaluations[target][group] = bundle

# The same successful pair must use exactly the same observations, not just counts.
for target, groups in comparison_evaluations.items():
    full_fits = groups[reference_group]["fits"]
    for group, bundle in groups.items():
        for key, fit in bundle["fits"].items():
            if key in full_fits and not fit["predictions"].index.equals(full_fits[key]["predictions"].index):
                raise ValueError(f"Observation mismatch: {target}, {group}, {key}")
comparison_scores = {
    target: pd.concat({group: bundle["summary"] for group, bundle in groups.items()}, names=["group"])
    for target, groups in comparison_evaluations.items()
}
comparison_results = {
    "settings": comparison_settings, "evaluations": comparison_evaluations,
    "scores": comparison_scores, "eligibility": comparison_eligibility,
    "paired_to_all": {}, "pairwise": {},
}
for target, summary in comparison_scores.items():
    print(target)
    display(summary.groupby(["group", "status"]).size().unstack(fill_value=0))

## 4. Matched comparisons
Points and mean bars use the neuron-type palette above. Panels distinguish
tracking/playback. Heatmap cell colors encode row-minus-column score differences;
colored labels identify the populations. Every comparison reports its own number
of matched sessions. Sensory comparisons have few sessions at the default cutoff.
Replot these cells after loading saved results; fitting is not needed for recoloring.

In [ ]:
comparison_figures = {}
for target, summary in comparison_scores.items():
    score = "balanced_accuracy" if target == "initial_direction" else LINEAR_SCORE
    fig, axes, matched = evaluation.plot_neuron_group_comparison(
        summary, score=score, reference_group=reference_group, group_colors=GROUP_COLORS,
    )
    fig.suptitle(f"{target} | {score} | {comparison_settings['activity_mode']} | same-session comparisons")
    comparison_results["paired_to_all"][target] = matched
    comparison_figures[(target, "full_session")] = fig
    plt.show()
    fig_pairs, axes_pairs, pair_summary = evaluation.plot_neuron_group_pairwise(summary, score=score, group_colors=GROUP_COLORS)
    fig_pairs.suptitle(f"{target}: paired mean score differences | {comparison_settings['activity_mode']}")
    comparison_results["pairwise"][target] = pair_summary
    comparison_figures[(target, "pairwise")] = fig_pairs
    plt.show()
    display(pair_summary)
comparison_results["settings"]["linear_score"] = LINEAR_SCORE
comparison_results["settings"]["group_colors"] = GROUP_COLORS.copy()

## 5. Save or reload
`pca_groups.pkl` stores each population's models, coordinates and selected unit
IDs. The evaluation pickle stores scores, fitted regressions/classifiers, original
prediction IDs, eligibility and paired comparisons. The run name identifies
preprocessing; the evaluation filename identifies PCs, destination filter and C.
The existing comparison output locations are retained.

In [ ]:
cs = comparison_settings
comparison_name = (
    f"neuron_group_comparison_min{cs['min_neurons']}_analysis_"
    f"{cs['analysis_window_s'][0]:g}_{cs['analysis_window_s'][1]:g}_{cs['N_BINS']}bins_{cs['activity_mode']}"
)
if cs["baseline_window_s"] is not None:
    comparison_name += f"_{cs['baseline_window_s'][0]:g}_{cs['baseline_window_s'][1]:g}"
# Include the chosen group set so optional subsets cannot overwrite the default comparison.
if cs["groups"][1:] != list(category_to_cluster):
    comparison_name += "_groups_" + "-".join(group.lower() for group in cs["groups"][1:])
comparison_dir = premotor_dir / "runs" / comparison_name
cutoff = cs["destination_min_displacement"]
filter_tag = "all_bouts" if cutoff is None else f"absdelta_gt{cutoff:g}"
comparison_path = comparison_dir / "evaluation" / (
    f"comparison_{'-'.join(cs['columns'])}_final_{filter_tag}_C{cs['direction_C']:g}_pooled.pkl"
)
if SAVE_RESULTS:
    comparison_path.parent.mkdir(parents=True, exist_ok=True)
    pd.to_pickle(comparison_pca_groups, comparison_dir / "pca_groups.pkl")
    pd.to_pickle(comparison_results, comparison_path)
    print(f"Saved comparison PCAs to {comparison_dir / 'pca_groups.pkl'}")
    print(f"Saved models, scores, matched differences and settings to {comparison_path}")
else:
    print("Comparison results remain in memory; SAVE_RESULTS is False.")

In [ ]:
# To recolor/replot a saved run, execute imports/settings, set the saved path,
# then run this snippet followed by the plots cell. No fitting is required.
# comparison_results = pd.read_pickle(comparison_path)
# comparison_settings = comparison_results["settings"]
# comparison_scores = comparison_results["scores"]
# LINEAR_SCORE = comparison_settings.get("linear_score", "r2_adjusted")
# reference_group = "All neurons"